# PhysioGraphSleep — Training Pipeline

Single-channel EEG sleep staging with heterogeneous graph neural networks.

**Target**: Sleep-EDF-20 — 20 subjects, 5-class (W/N1/N2/N3/REM)

## Bu notebook'un ilkeleri
- **Tüm dosyalar `physiographsleep/` klasörünün İÇİNDE** üretilir (cache, checkpoints, logs, outputs).
- **WandB** opsiyonel; `WANDB_ENABLED=True` yapıp `WANDB_API_KEY` ortam değişkenini ayarlayın.
- **Live plots** her epoch sonunda otomatik güncellenir (matplotlib).
- **Detaylı tablo**: train/val loss + accuracy + macro-F1 + per-class F1 her epoch için yazdırılır.

## Kullanım
1. Colab → Runtime → Change runtime type → **GPU (T4/L4/A100)**
2. Hücreleri sırayla çalıştırın.


## 1. Setup & Repo Clone

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*Channels contain different.*")
warnings.filterwarnings("ignore", message=".*Highpass cutoff frequency.*")
warnings.filterwarnings("ignore", message=".*Lowpass cutoff frequency.*")
os.environ["MNE_LOGGING_LEVEL"] = "ERROR"

# === Yapılandırma flagları ===
WANDB_ENABLED = True  # True yapıp aşağıdaki API key cell'ini doldurun
WANDB_PROJECT = "physiographsleep"
WANDB_RUN_NAME = "sleepedf20"

# Colab /dev/shm genişlet (multi-worker DataLoader için)
if "google.colab" in sys.modules:
    !df -h /dev/shm 2>/dev/null
    !sudo mount -o remount,size=2G /dev/shm 2>/dev/null
    !df -h /dev/shm 2>/dev/null


In [ ]:
# Repo clone (Colab) veya local repo path tespiti
REPO_URL = "https://github.com/0nur0duncu/physiographsleep.git"

if "google.colab" in sys.modules:
    PROJECT_DIR = "/content/physiographsleep"
    if os.path.exists(PROJECT_DIR):
        !cd {PROJECT_DIR} && git pull
        print(f"Repo güncellendi: {PROJECT_DIR}")
    else:
        !git clone {REPO_URL} {PROJECT_DIR}
        print(f"Repo klonlandı: {PROJECT_DIR}")
    PARENT_DIR = "/content"
else:
    # Local: notebook physiographsleep/ içinde, parent'ı sys.path'e ekle
    PROJECT_DIR = os.path.dirname(os.path.abspath("."))
    if os.path.basename(os.getcwd()) != "physiographsleep":
        PROJECT_DIR = os.path.abspath(".")
    PARENT_DIR = os.path.dirname(PROJECT_DIR)
    print(f"Local repo: {PROJECT_DIR}")

if PARENT_DIR not in sys.path:
    sys.path.insert(0, PARENT_DIR)
os.chdir(PARENT_DIR)
print(f"Working dir: {os.getcwd()}")
print(f"Project root: {PROJECT_DIR}")


In [ ]:
# Bağımlılıklar
!pip install -q -r {PROJECT_DIR}/requirements.txt
if WANDB_ENABLED:
    !pip install -q wandb


In [ ]:
import torch
import numpy as np
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


## 2. Configuration

> Tüm yollar `PROJECT_DIR` (= `physiographsleep/` klasörü) altında. Hiçbir şey bu klasörün dışına yazılmaz.

In [ ]:
import os
import sys
from pathlib import Path

# === PROJECT_DIR (Colab vs local) ===
# Repo yapısı:
#   <parent>/physiographsleep/          ← git repo root + Python package
#            ├── configs/ models/ training/ ...
#            ├── main.ipynb  (bu notebook)
#            └── __init__.py
# Colab'da: /content/physiographsleep   (git clone ile oluşur)
# Local'de: cwd zaten neurographTdraft/, physiographsleep/ alt klasör.

if "google.colab" in sys.modules:
    PROJECT_DIR = Path("/content/physiographsleep")
    IS_COLAB = True
else:
    cwd = Path.cwd()
    # Local: workspace root (neurographTdraft/) altında physiographsleep/ var mı?
    if (cwd / "physiographsleep" / "__init__.py").exists():
        PROJECT_DIR = cwd / "physiographsleep"
    elif (cwd / "__init__.py").exists() and cwd.name == "physiographsleep":
        PROJECT_DIR = cwd
    else:
        PROJECT_DIR = cwd
    IS_COLAB = False

# Repo root = package root olduğu için __init__.py ile doğrula
assert (PROJECT_DIR / "__init__.py").exists(), (
    f"physiographsleep package not found at {PROJECT_DIR}\n"
    f"Expected __init__.py in {PROJECT_DIR}"
)
PARENT_DIR = str(PROJECT_DIR.parent)
if PARENT_DIR not in sys.path:
    sys.path.insert(0, PARENT_DIR)
print(f"PROJECT_DIR = {PROJECT_DIR}  (Colab={IS_COLAB})")

# === imports ===
from physiographsleep.configs import ExperimentConfig
from physiographsleep.utils.reproducibility import get_device, set_seed

# === Channel mode (1ch EEG vs 2ch EEG+EOG) ===
USE_EOG = False  # True = 2 kanal (EEG Fpz-Cz + EOG horizontal); False = 1 kanal

# === WandB toggles ===
WANDB_ENABLED = True
WANDB_PROJECT = "neurographTdraft"
WANDB_RUN_NAME = "physiographsleep_sota_revert"

# === Output paths (PROJECT_DIR İÇİNDE — Drive değil) ===
OUTPUTS_DIR = PROJECT_DIR / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

# === Build config ===
config = ExperimentConfig()
# Colab'da /content/data/sleepedf20 (kalıcı olmaz ama hızlı); local'de proje altı.
if IS_COLAB:
    config.data.data_dir = "/content/data/sleepedf20"
    config.data.cache_dir = "/content/cache"
else:
    config.data.data_dir = str(PROJECT_DIR / "data" / "sleepedf20")
    config.data.cache_dir = str(PROJECT_DIR / "cache")
config.train.checkpoint_dir = str(PROJECT_DIR / "checkpoints")
config.train.log_dir = str(PROJECT_DIR / "logs")

# Channel mode patch
if USE_EOG:
    config.data.use_eog = True
    config.model.waveform.in_channels = config.data.num_input_channels  # 2

# Eğitim hiperparametreleri (defaults zaten dengeli; aşağıdaki override'lar
# Sleep-EDF-20 için tarihi kalibrasyon).
# Önceki başarılı run (commit 9511d6d): raw MF1=0.796 / biased=0.808 / HMM=0.813.
config.train.epochs = 60
config.train.lr = 1e-3
config.train.n1_boost = 2.0
# patience varsayılan 10 (config tarafında ayarlandı).

# Pathway / λ-fusion / N1-Mixup ablation toggle'ları BİR SONRAKİ HÜCREDE
# yönetilir. Varsayılan (`True`) → yeni scGraPhT defaults; `False` →
# SOTA-benzeri referans konfigürasyon (commit 9511d6d).

# === Performans toggles ===
USE_TORCH_COMPILE = False

set_seed(config.seed)
device = get_device("auto")


In [ ]:
# ============================================================
# Ablation / SOTA-revert toggles
# ------------------------------------------------------------
# Git arkeolojisi (commit 9511d6d, Apr 18): raw MF1=0.796 / biased=0.808
# / HMM=0.813 sonuçlarını veren referans konfigürasyonda ŞU ÜÇ FEATURE
# KAPALIYDI. Sonraki commit (e7bf922) hepsini aynı anda default AÇIK
# yaptı + wd/drop_path/patience'ı da değiştirdi → regresyon.
#
# Tek değişken A/B yapabilmek için aşağıdaki üç anahtarı bağımsız
# açıp kapatabilirsin:
#   False → feature kapalı (SOTA-benzeri)
#   True  → feature açık (yeni default)
# ============================================================
USE_PATHWAY  = False   # hetero → homo → all-edges pipeline
USE_FUSION   = False   # λ-fusion (auxiliary transformer head)
USE_N1_MIXUP = False   # N1-targeted Mixup augmentation

if not USE_PATHWAY:
    config.model.graph.edge_pathways = None
if not USE_FUSION:
    config.model.fusion = None
if not USE_N1_MIXUP:
    config.train.n1_mixup = None

# ============================================================
# Final özet — ablation sonrası efektif konfigürasyon
# ============================================================
print("=" * 60)
print("PATHS")
print("=" * 60)
print(f"  data_dir:       {config.data.data_dir}")
print(f"  cache_dir:      {config.data.cache_dir}")
print(f"  checkpoint_dir: {config.train.checkpoint_dir}")
print(f"  log_dir:        {config.train.log_dir}")
print(f"  outputs_dir:    {OUTPUTS_DIR}")
print("=" * 60)
print(f"Device:           {device}")
print(f"Subjects:         {config.data.num_subjects} (train={config.data.train_subjects}, val={config.data.val_subjects})")
print(f"Batch size:       {config.data.batch_size}")
print(f"Num workers:      {config.data.num_workers}")
print(f"Wake-trim (min):  {config.data.wake_trim_minutes}")
print(f"Channel mode:     {'EEG Fpz-Cz + EOG horizontal (2ch)' if USE_EOG else 'EEG Fpz-Cz only (1ch)'}")
print(f"WaveformStem in:  {config.model.waveform.in_channels} channel(s)")
print(f"torch.compile:    {USE_TORCH_COMPILE}")
print(f"Epochs:           {config.train.epochs}  (early stop patience={config.train.patience})")
print(f"LR / N1 boost:    {config.train.lr} / {config.train.n1_boost}x")
print(f"Graph layers/heads: {config.model.graph.num_layers} / {config.model.graph.num_heads}  (DropPath={config.model.graph.drop_path})")
print("=" * 60)
print("ABLATION TOGGLES")
print("=" * 60)
print(f"  pathway:          {'ON ' if USE_PATHWAY  else 'OFF'}  → edge_pathways={config.model.graph.edge_pathways}")
fusion_str = (
    f"init_λ={config.model.fusion.init_lambda} detach_gnn={config.model.fusion.detach_gnn_for_lambda}"
    if config.model.fusion is not None else "DISABLED"
)
print(f"  λ-fusion:         {'ON ' if USE_FUSION   else 'OFF'}  → {fusion_str}")
mix_str = (
    f"prob={config.train.n1_mixup.prob} α={config.train.n1_mixup.alpha}"
    if config.train.n1_mixup is not None else "DISABLED"
)
print(f"  N1-Mixup:         {'ON ' if USE_N1_MIXUP else 'OFF'}  → {mix_str}")
print(f"Label smoothing:  {config.train.loss.label_smoothing}  | WD: {config.train.optimizer.weight_decay}")


In [ ]:
# WandB init (opsiyonel)
wandb_run = True
if WANDB_ENABLED:
    import wandb
    # Colab: secrets'tan API key alın veya kullanıcı interaktif login olur
    try:
        from google.colab import userdata  # type: ignore
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    except Exception:
        pass  # Local: WANDB_API_KEY env veya wandb login

    wandb_run = wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        dir=OUTPUTS_DIR,
        config={
            "num_subjects": config.data.num_subjects,
            "batch_size": config.data.batch_size,
            "seq_len": config.data.seq_len,
            "epochs": config.train.epochs,
            "lr": config.train.lr,
            "n1_boost": config.train.n1_boost,
            "patience": config.train.patience,
        },
    )
    print(f"WandB run: {wandb_run.url}")
else:
    print("WandB devre dışı (WANDB_ENABLED=False)")


## 3. Veri Yükleme + Sınıf Dağılımı

In [ ]:
from physiographsleep.data.loader import load_sleep_edf

data = load_sleep_edf(config.data)

STAGE_NAMES = ["W", "N1", "N2", "N3", "REM"]
print("Split sizes:")
for split_name, split_data in data.items():
    epochs = split_data["epochs"]
    labels = split_data["labels"]
    spectral = split_data.get("spectral")
    print(f"  {split_name:5s}: {len(labels):6d} epochs | shape={epochs.shape} | spectral={spectral.shape if spectral is not None else 'N/A'}")

print("\nTrain class distribution:")
train_labels = data["train"]["labels"]
counts = np.bincount(train_labels, minlength=5)
total = len(train_labels)
for i, name in enumerate(STAGE_NAMES):
    pct = 100 * counts[i] / total
    bar = "█" * int(pct / 2)
    print(f"  {name:3s}: {counts[i]:6d} ({pct:5.1f}%) {bar}")


## 4. DataLoaders

In [ ]:
from torch.utils.data import DataLoader
from physiographsleep.data.dataset import SleepEDFDataset
from physiographsleep.data.transforms import SleepTransforms

train_transform = SleepTransforms(config.data)

train_ds = SleepEDFDataset(
    data["train"]["epochs"], data["train"]["labels"],
    config=config.data, transform=train_transform,
    spectral=data["train"].get("spectral"),
)
val_ds = SleepEDFDataset(
    data["val"]["epochs"], data["val"]["labels"],
    config=config.data,
    spectral=data["val"].get("spectral"),
)

val_loader = DataLoader(
    val_ds, batch_size=config.data.batch_size,
    shuffle=False, num_workers=config.data.num_workers,
    pin_memory=True,
    persistent_workers=config.data.num_workers > 0,
    prefetch_factor=4 if config.data.num_workers > 0 else None,
)

print(f"Train: {len(train_ds)} samples")
print(f"Val:   {len(val_ds)} samples, {len(val_loader)} batches")


## 5. Model + Sanity Check

In [ ]:
from physiographsleep.models.physiographsleep import PhysioGraphSleep

model = PhysioGraphSleep(config.model)
param_counts = model.count_parameters()

print("Parameter counts:")
for name, count in param_counts.items():
    print(f"  {name:20s}: {count:>10,}")

# Sanity forward
batch = next(iter(val_loader))
with torch.no_grad():
    model.eval()
    sig = batch["signal"][:2]
    spec = batch["spectral"][:2] if "spectral" in batch else torch.zeros(2, config.data.seq_len, 5, 42)
    out = model(sig, spec)
print(f"\nSanity forward: signal={tuple(sig.shape)} → stage={tuple(out['stage'].shape)} OK")

# torch.compile (opsiyonel — ilk epoch +60s, sonraki epoch'lar 1.3-2x hızlı)
if USE_TORCH_COMPILE and torch.cuda.is_available():
    print("\ntorch.compile aktif (ilk derleme uzun sürebilir)...")
    model = torch.compile(model, mode="reduce-overhead", dynamic=False)


## 6. Training — Live Plots + WandB + Detailed Epoch Table

Bu hücre `Trainer.callback` hook'unu kullanarak her epoch sonunda:
- Detaylı tablo satırı yazdırır (train loss/acc/MF1, val acc/MF1/κ, per-class F1)
- 4 panelli live plot çizer (loss, accuracy, macro-F1, per-class val F1)
- WandB'ye metrikleri loglar (etkinse)

In [ ]:
import gc
import json
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

from physiographsleep.models.losses import MultiTaskLoss
from physiographsleep.data.spectral import SpectralFeatureExtractor
from physiographsleep.training.trainer import Trainer
from physiographsleep.utils.logging_utils import setup_logger

# Free GPU memory
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, {torch.cuda.memory_reserved()/1e9:.2f} GB reserved")

# NOTE: class_weights'i KASITLI OLARAK kaldırdık.
# Sampler zaten N1'i 2x boost'luyor (WeightedRandomSampler with n1_boost=2.0).
# Aynı anda inverse-frequency class_weights (N1=2.29x) uygulamak N1'i
# efektif ~4.6x over-weight ediyor ve F1_N1'yi platoya sokuyor.
# Tek mekanizma: sampler-level N1 boost. Loss uniform kalır.
loss_fn = MultiTaskLoss(config.train.loss)
spectral_ext = SpectralFeatureExtractor(config.data)
logger = setup_logger(log_dir=config.train.log_dir)
print("Loss: FocalLoss(gamma=2.0, label_smoothing=0.1) — class_weights disabled (sampler handles N1)")


In [ ]:
# ============================================================
# Training history + per-epoch callback
# ============================================================
HISTORY = {"epoch": [],
           "train_loss": [], "val_loss": [],
           "train_acc": [], "train_mf1": [],
           "val_acc": [], "val_mf1": [], "val_mf1_gnn": [], "val_kappa": [],
           "val_f1_W": [], "val_f1_N1": [], "val_f1_N2": [],
           "val_f1_N3": [], "val_f1_REM": []}

HEADER_PRINTED = {"v": False}
def _print_header():
    if HEADER_PRINTED["v"]:
        return
    cols = ("Epoch", "TrLoss", "VlLoss",
            "TrAcc", "TrMF1", "VlAcc", "VlMF1", "GnnMF1", "Kappa",
            "F1_W", "F1_N1", "F1_N2", "F1_N3", "F1_REM")
    print(" | ".join(f"{c:>7s}" for c in cols))
    print("-" * (10 * len(cols)))
    HEADER_PRINTED["v"] = True

def _draw_live_plot():
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    epochs_x = list(range(len(HISTORY["epoch"])))

    axes[0, 0].plot(epochs_x, HISTORY["train_loss"], "-o", ms=3, label="train")
    axes[0, 0].plot(epochs_x, HISTORY["val_loss"],   "-s", ms=3, label="val")
    axes[0, 0].set_title("Loss"); axes[0, 0].set_xlabel("epoch"); axes[0, 0].grid(alpha=0.3)
    axes[0, 0].legend()

    axes[0, 1].plot(epochs_x, HISTORY["train_acc"], "-o", ms=3, label="train")
    axes[0, 1].plot(epochs_x, HISTORY["val_acc"],   "-s", ms=3, label="val")
    axes[0, 1].set_title("Accuracy"); axes[0, 1].set_ylim(0, 1); axes[0, 1].grid(alpha=0.3)
    axes[0, 1].legend()

    axes[1, 0].plot(epochs_x, HISTORY["train_mf1"], "-o", ms=3, label="train")
    axes[1, 0].plot(epochs_x, HISTORY["val_mf1"],   "-s", ms=3, label="val (fused)")
    # GNN-only MF1 — fusion kapalıyken val_mf1 ile aynı; açıkken ayrık eğri
    if any(v is not None for v in HISTORY["val_mf1_gnn"]):
        gnn_y = [v if v is not None else float("nan") for v in HISTORY["val_mf1_gnn"]]
        axes[1, 0].plot(epochs_x, gnn_y, "-^", ms=3, label="val (gnn-only)")
    axes[1, 0].set_title("Macro-F1"); axes[1, 0].set_ylim(0, 1); axes[1, 0].grid(alpha=0.3)
    axes[1, 0].legend()

    for cls in ["W", "N1", "N2", "N3", "REM"]:
        axes[1, 1].plot(epochs_x, HISTORY[f"val_f1_{cls}"], "-o", ms=3, label=cls)
    axes[1, 1].set_title("Val per-class F1"); axes[1, 1].set_ylim(0, 1); axes[1, 1].grid(alpha=0.3)
    axes[1, 1].legend(ncol=5, fontsize=8)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, "training_curves.png"), dpi=80, bbox_inches="tight")
    clear_output(wait=True)
    display(fig)
    plt.close(fig)


def epoch_callback(epoch, train_loss, val_loss, train_metrics, val_metrics):
    HISTORY["epoch"].append(epoch)
    HISTORY["train_loss"].append(train_loss)
    HISTORY["val_loss"].append(val_loss)
    HISTORY["train_acc"].append(train_metrics.get("accuracy", 0.0))
    HISTORY["train_mf1"].append(train_metrics.get("macro_f1", 0.0))
    HISTORY["val_acc"].append(val_metrics.get("accuracy", 0.0))
    HISTORY["val_mf1"].append(val_metrics.get("macro_f1", 0.0))
    HISTORY["val_mf1_gnn"].append(val_metrics.get("macro_f1_gnn"))  # None ise fusion kapalı
    HISTORY["val_kappa"].append(val_metrics.get("kappa", 0.0))
    for cls in ["W", "N1", "N2", "N3", "REM"]:
        HISTORY[f"val_f1_{cls}"].append(val_metrics.get(f"f1_{cls}", 0.0))

    _print_header()
    gnn_mf1 = val_metrics.get("macro_f1_gnn")
    gnn_str = f"{gnn_mf1:>7.4f}" if gnn_mf1 is not None else "   N/A "
    row = (
        f"{epoch:>7d}",
        f"{train_loss:>7.4f}", f"{val_loss:>7.4f}",
        f"{train_metrics.get('accuracy', 0):>7.4f}",
        f"{train_metrics.get('macro_f1', 0):>7.4f}",
        f"{val_metrics.get('accuracy', 0):>7.4f}",
        f"{val_metrics.get('macro_f1', 0):>7.4f}",
        gnn_str,
        f"{val_metrics.get('kappa', 0):>7.4f}",
        f"{val_metrics.get('f1_W', 0):>7.4f}",
        f"{val_metrics.get('f1_N1', 0):>7.4f}",
        f"{val_metrics.get('f1_N2', 0):>7.4f}",
        f"{val_metrics.get('f1_N3', 0):>7.4f}",
        f"{val_metrics.get('f1_REM', 0):>7.4f}",
    )
    print(" | ".join(row))

    if wandb_run is not None:
        log = {
            "train/loss": train_loss,
            "val/loss": val_loss,
            "train/acc": train_metrics.get("accuracy", 0),
            "train/macro_f1": train_metrics.get("macro_f1", 0),
            "val/acc": val_metrics.get("accuracy", 0),
            "val/macro_f1": val_metrics.get("macro_f1", 0),
            "val/kappa": val_metrics.get("kappa", 0),
        }
        for cls in ["W", "N1", "N2", "N3", "REM"]:
            log[f"val/f1_{cls}"] = val_metrics.get(f"f1_{cls}", 0)
        # λ-fusion açıkken pure-GNN baseline'ı da logla (diagnostic)
        if gnn_mf1 is not None:
            log["val/macro_f1_gnn"] = gnn_mf1
            log["val/acc_gnn"] = val_metrics.get("accuracy_gnn", 0)
        # step=epoch + commit=True → canlı web dashboard güncellenir
        wandb_run.log(log, step=epoch, commit=True)

    _draw_live_plot()

    with open(os.path.join(OUTPUTS_DIR, "history.json"), "w") as f:
        json.dump(HISTORY, f, indent=2)


trainer = Trainer(
    model=model,
    loss_fn=loss_fn,
    train_dataset=train_ds,
    train_labels=data["train"]["labels"],
    val_loader=val_loader,
    config=config.train,
    data_config=config.data,
    device=device,
    spectral_extractor=spectral_ext,
    callback=epoch_callback,
)
print("Trainer hazır. Sonraki hücre eğitimi başlatır.")


In [ ]:
# Auto-resume + run training
from physiographsleep.utils.io_utils import load_checkpoint
from physiographsleep.evaluation.metrics import MetricsCalculator

ckpt_path = os.path.join(config.train.checkpoint_dir, config.train.checkpoint_name)
if os.path.exists(ckpt_path):
    ckpt = load_checkpoint(ckpt_path, device)
    model.load_state_dict(ckpt["model"])
    prev_mf1 = ckpt["metrics"]["macro_f1"]
    print(f"Mevcut checkpoint bulundu: {ckpt_path} (val MF1={prev_mf1:.4f})")
    print("→ Yeniden eğitmek istemiyorsan bu hücreyi atla, post-processing'e geç.")
    print("→ Yeniden eğitmek için checkpoint dosyasını silip bu hücreyi tekrar çalıştır.")

best_metrics = trainer.train()
print("\n" + "=" * 50)
print("TRAINING COMPLETE")
print("=" * 50)
print(MetricsCalculator.format_report(best_metrics))


## 7. Test Eval + HMM Post-processing + Confusion Matrices

In [ ]:
import torch.nn.functional as F

from physiographsleep.training.evaluator import Evaluator
from physiographsleep.evaluation.metrics import MetricsCalculator
from physiographsleep.evaluation.postprocessing import HMMPostProcessor, LogitBiasOptimizer
from physiographsleep.evaluation.visualization import plot_confusion_matrix

# En iyi checkpoint'i yükle
ckpt_path = os.path.join(config.train.checkpoint_dir, config.train.checkpoint_name)
if os.path.exists(ckpt_path):
    ckpt = load_checkpoint(ckpt_path, device)
    model.load_state_dict(ckpt["model"])
    print(f"Yüklendi: {ckpt_path} (val MF1={ckpt['metrics']['macro_f1']:.4f})")
else:
    print("Checkpoint bulunamadı, mevcut model state kullanılıyor")

evaluator = Evaluator(device)

if "test" in data:
    test_ds = SleepEDFDataset(
        data["test"]["epochs"], data["test"]["labels"],
        config=config.data,
        spectral=data["test"].get("spectral"),
    )
    test_loader = DataLoader(
        test_ds, batch_size=config.data.batch_size,
        shuffle=False, num_workers=config.data.num_workers,
    )

    # Test ve val logits
    test_metrics, test_logits, test_labels_arr = evaluator.evaluate(
        model, test_loader, spectral_ext, return_logits=True,
    )
    val_loader_eval = DataLoader(
        val_ds, batch_size=config.data.batch_size,
        shuffle=False, num_workers=config.data.num_workers,
    )
    _, val_logits, val_labels_arr = evaluator.evaluate(
        model, val_loader_eval, spectral_ext, return_logits=True,
    )

    calc = MetricsCalculator()
    results = []  # [(name, predictions, metrics), ...]

    # [1] Raw argmax — baseline
    preds_raw = test_logits.argmax(axis=1)
    raw_m = calc.compute_all(test_labels_arr, preds_raw)
    results.append(("raw", preds_raw, raw_m))

    # [2] Logit-bias (val set üzerinde Nelder-Mead ile macro-F1 maksimize):
    # AttnSleep / SleepTransformer'ın class-imbalance düzeltmesi.
    bias_opt = LogitBiasOptimizer(num_classes=5)
    bias_opt.fit(val_logits, val_labels_arr)
    preds_biased = bias_opt.apply(test_logits)
    biased_m = calc.compute_all(test_labels_arr, preds_biased)
    results.append(("biased", preds_biased, biased_m))

    # [3] HMM posterior Viterbi smoothing (DeepSleepNet/AttnSleep standart):
    # Transition matrix train labels'tan, log-softmax(bias-adjusted logits)
    # üzerinden Viterbi decoding.
    adjusted_logits = test_logits + bias_opt.bias
    log_post = F.log_softmax(torch.from_numpy(adjusted_logits), dim=-1).numpy()
    hmm = HMMPostProcessor(num_classes=5).fit(data["train"]["labels"])
    preds_hmm = hmm.smooth_posteriors(log_post)
    hmm_m = calc.compute_all(test_labels_arr, preds_hmm)
    results.append(("hmm", preds_hmm, hmm_m))

    print("\n" + "=" * 50)
    print("TEST RESULTS")
    print("=" * 50)
    prev = None
    for name, _, m in results:
        delta = f"  ({m['macro_f1']-prev['macro_f1']:+.4f})" if prev else ""
        print(f"\n[{name}] MF1={m['macro_f1']:.4f}{delta}")
        print(MetricsCalculator.format_report(m))
        prev = m

    # Confusion matrices → outputs/
    for name, preds, m in results:
        cm = MetricsCalculator.confusion_matrix(test_labels_arr, preds)
        plot_confusion_matrix(
            cm,
            save_path=os.path.join(OUTPUTS_DIR, f"cm_{name}.png"),
            title=f"{name} (MF1={m['macro_f1']:.3f})",
        )
    print(f"\nConfusion matrices kaydedildi: {OUTPUTS_DIR}")

    # Final dump
    final_metrics = {name: m for name, _, m in results}
    final_path = os.path.join(OUTPUTS_DIR, "final_model.pt")
    torch.save({
        "model": model.state_dict(),
        "config": {"num_subjects": config.data.num_subjects, "batch_size": config.data.batch_size,
                   "seq_len": config.data.seq_len},
        "metrics": final_metrics,
        "param_counts": param_counts,
    }, final_path)
    print(f"Final model: {final_path}  ({param_counts['total']:,} params)")

    if wandb_run is not None:
        for name, _, m in results:
            wandb_run.summary[f"test_{name}_mf1"] = m["macro_f1"]
            wandb_run.summary[f"test_{name}_acc"] = m["accuracy"]
            for cls in ["W", "N1", "N2", "N3", "REM"]:
                wandb_run.summary[f"test_{name}_f1_{cls}"] = m.get(f"f1_{cls}", 0)
        wandb_run.save(os.path.join(OUTPUTS_DIR, "*.png"))
        wandb_run.save(os.path.join(OUTPUTS_DIR, "history.json"))
        wandb_run.finish()
        print("WandB run kapatıldı")
else:
    print("Test split yok")
